<div dir="rtl" align="right">

# مقایسه طبقه‌بندها، چندکلاسه و داده نامتوازن

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## جدول راهنما
| مدل | مقیاس‌بندی | مرز | احتمال خوب | سرعت پیش‌بینی | داده بزرگ |
|---|---|---|---|---|---|
| لجستیک | **بله** | خطی | عالی | خیلی سریع | عالی |
| KNN | **بله** | نامنظم | متوسط | کند | ضعیف |
| بیز ساده | خیر | ساده | ضعیف | خیلی سریع | عالی |
| درخت | خیر | پله‌ای | ضعیف | خیلی سریع | خوب |
| جنگل | خیر | پله‌ای نرم | خوب | متوسط | خوب |
| بوستینگ | خیر | پیچیده | خوب | سریع | عالی |
| SVM کرنل | **بله** | منحنی صاف | با تنظیم | متوسط | ضعیف |

[[lab:cls_all]]

## مسائل چندکلاسه
مدل‌های دودویی (مثل SVM) با دو راهبرد چندکلاسه می‌شوند:

| راهبرد | تعداد مدل برای K کلاس | ایده |
|---|---|---|
| یکی در برابر بقیه (OvR) | K | هر کلاس در برابر همه |
| یکی در برابر یکی (OvO) | K(K−1)/2 | هر جفت کلاس؛ رأی‌گیری |
| Softmax | ۱ | لجستیک چندجمله‌ای مستقیم |



</div>

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import cross_val_score
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)   # 10 classes
for name, m in [("OvR  (10 models)", OneVsRestClassifier(LinearSVC(C=0.05))),
                ("OvO  (45 models)", OneVsOneClassifier(LinearSVC(C=0.05))),
                ("Softmax (1 model)", LogisticRegression(max_iter=2000))]:
    s = cross_val_score(make_pipeline(StandardScaler(), m), X, y, cv=5).mean()
    print(f"{name}: accuracy = {s:.3f}")

<div dir="rtl" align="right">



## داده نامتوازن
وقتی یک کلاس خیلی کمیاب است (تقلب ۱٪):

| راه حل | چطور |
|---|---|
| معیار مناسب | F1، Recall، Precision-Recall AUC به جای Accuracy |
| وزن کلاس | `class_weight="balanced"` |
| تغییر آستانه | آستانه کمتر از ۰٫۵ برای Recall بیشتر |
| نمونه‌گیری | کم کردن کلاس بزرگ یا افزایش کلاس کوچک (SMOTE در imbalanced-learn) |
| تقسیم طبقه‌ای | `stratify=y` و `StratifiedKFold` |



</div>

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X, y = make_classification(5000, n_features=10, weights=[0.97], flip_y=0.01, random_state=0)
print("Share of class 1:", y.mean().round(3))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)
for cw in [None, "balanced"]:
    m = LogisticRegression(class_weight=cw, max_iter=1000).fit(X_tr, y_tr)
    print("class_weight =", cw)
    print(classification_report(y_te, m.predict(X_te), digits=3))

<div dir="rtl" align="right">



> هرگز قبل از تقسیم داده نمونه‌گیری مجدد نکن؛ فقط روی داده آموزش و داخل اعتبارسنجی.

</div>